# K-means Clustering
## Using Snowpark Python and Scikit-Learn
### Overview
This notebook finds a set of cluster centers for customer addresses. 

Note that, for simplicity, the customer addresses are already resolved to geolocations (latitude and longitude). A production system could include a call to an external service that performs this geolocation on new customer addresses.

Steps:
- Setup
- Load and Explore Data
- Cluster to Find 3 Central Locations
- Save Cluster Centers to Snowflake
- Areas for Further Investigation/Work

### Setup

## My overview

- Unsupervised learning: K-Means for clustering
- Save model parameters to Snowflake table
- Create snowflake UDF to use model for inferencing within Snowflake: NB This exercise doesn't actually put any of the python code into the UDF - rather the UDF uses the results of the python k means clustering process, with build in geospatial functions, to return close pickup location (already generated from python clustering script) for a customer. **So this isn't an example of actually "productionising" python code in snowflake**

## MyNote: Generate data, as course account with data was unavailable
```sql
--Generate longitude & lattitude co-ordinates within max_distance_meters of center location (e.g. MCR airport): 

create table CREDITRISK_DEMO.public.locations_MCR_aiport as

WITH parameters AS (
    SELECT 
        53.365  AS center_lat,
        -2.272  AS center_lon,
        52.915  AS min_lat,
        53.815  AS max_lat,
        -2.972  AS min_lon,
        -1.572  AS max_lon,
        50000   AS max_distance_meters
),
random_points AS (
    SELECT 
        SEQ4() AS id,

        --original CGPT version - doesn't work
        -- min_lat + (max_lat - min_lat) * RANDOM(100000, 999999) / 999999.0 AS latitude,
        -- min_lon + (max_lon - min_lon) * RANDOM(100000, 999999) / 999999.0 AS longitude

        -- my corrected version
        min_lat + (max_lat - min_lat) * UNIFORM(100000, 999999, RANDOM()) / 999999.0 AS latitude,
        min_lon + (max_lon - min_lon) * UNIFORM(100000, 999999, RANDOM()) / 999999.0 AS longitude
        
    FROM parameters,
    TABLE(GENERATOR(ROWCOUNT => 500))
)
,
with_distance AS (
    SELECT 
        id,
        latitude,
        longitude,
        ST_DISTANCE(
            TO_GEOGRAPHY(ST_POINT(longitude, latitude)),
            TO_GEOGRAPHY(ST_POINT(center_lon, center_lat))
        ) AS distance_to_airport_meters
    FROM random_points, parameters
)


SELECT 
    id, 
    latitude, 
    longitude, 
    distance_to_airport_meters
FROM with_distance
WHERE distance_to_airport_meters <= (SELECT max_distance_meters FROM parameters)
-- ORDER BY id;

select * 
from CREDITRISK_DEMO.public.locations_MCR_aiport

```

In [ ]:
# Install folium -- not included in this Python build
# pip install folium

In [28]:
import snowflake.snowpark
from snowflake.snowpark.session import Session

import folium

# config_dir = '/home/jovyan/.ssh'
config_dir = '/Users/richardkirk/.ssh'
configfile = config_dir + '/sf_config'

* Load configuration and connect to Snowflake

In [29]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

### Load and Explore Data

- Examine via a Snowpark DataFrame

In [30]:
# membersDF = session.table('data_science_db.raw.geo_members')
membersDF = session.table('CREDITRISK_DEMO.public.locations_MCR_aiport')

In [31]:
membersDF.schema.fields

[StructField('ID', LongType(), nullable=True),
 StructField('LATITUDE', DecimalType(19, 9), nullable=True),
 StructField('LONGITUDE', DecimalType(18, 9), nullable=True),
 StructField('DISTANCE_TO_AIRPORT_METERS', DoubleType(), nullable=True)]

In [32]:
membersDF.count()

455

In [33]:
membersDF.show(5)

---------------------------------------------------------------------
|"ID"  |"LATITUDE"    |"LONGITUDE"   |"DISTANCE_TO_AIRPORT_METERS"  |
---------------------------------------------------------------------
|0     |53.313150498  |-2.196564425  |7636.971381420159             |
|1     |53.510315495  |-2.377070805  |17593.487525918274            |
|2     |53.232296217  |-1.828942257  |32934.039192181575            |
|3     |53.400531586  |-1.841745270  |28808.43834493683             |
|4     |53.251132336  |-1.910541339  |27148.750804649895            |
---------------------------------------------------------------------



* For the purpose of modeling, we will keep only member_id, lat, and lon (latitude and longitude).

In [42]:
membersDF = membersDF.select('ID', 'LATITUDE', 'LONGITUDE')
membersDF.show(5)

--------------------------------------
|"ID"  |"LATITUDE"    |"LONGITUDE"   |
--------------------------------------
|0     |53.313150498  |-2.196564425  |
|1     |53.510315495  |-2.377070805  |
|2     |53.232296217  |-1.828942257  |
|3     |53.400531586  |-1.841745270  |
|4     |53.251132336  |-1.910541339  |
--------------------------------------



* Plot the members' home address locations.

In [35]:
def plot_points(df):
    
    # Map centered on MCR airport
    # m = folium.Map(location=[37.62, -122.365], zoom_start=7.5)
    m = folium.Map(location=[53.365, -2.272], zoom_start=7.5)
    
    # Obtain [LATITUDE, LONGITUDE] columns as list of Python Row objects to plot
    pointsLST = df.select('LATITUDE', 'LONGITUDE').collect()
    
    for point in pointsLST:
        folium.CircleMarker(location=[point['LATITUDE'], point['LONGITUDE']], \
                           radius=1) \
              .add_to(m)
    return(m)

In [36]:
plot_points(membersDF)

Members are located in the vicinity of the San Francisco (SFO) airport.

### Cluster to Find 3 Central Locations
Given these customer home address locations, find three general locations that center around clusters of customers. To compete in this market, we will consider providing a free or inexpensive luxury shuttle from each of these "transportation hubs" to the SFO airport.

In [38]:
# Fetch points to a Pandas DataFrame for training
pointsPDF = membersDF.select('LATITUDE', 'LONGITUDE').toPandas()

* Choose K-means for a clustering algorithm, and train the model to find three cluster centers using the available data.

In [41]:
# Error here: AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
# Seems due to old verison of numpy that is reqauired by snowflake-ml-python. 

from snowflake.ml.modeling.cluster import KMeans as KM
k = 3
kmeans = KM(n_clusters=k)

# Get an array of predictions
predictions = kmeans.fit_predict(pointsPDF)

# Convert to an array
predictions =predictions['OUTPUT_0'].to_numpy()

AttributeError: module 'numpy' has no attribute '_no_nep50_warning'

- Show the cluster centers

In [ ]:
# cluster_centers_ is not yet supported in Snowpark KMeans.
# So we revert back to sklearn:
kmeans.to_sklearn().cluster_centers_

- Score the data points with cluster membership, and show a few.

In [ ]:
import pandas as pd
predictionsPDF = pd.DataFrame(predictions, columns=['prediction'])
scored_pointsPDF = pointsPDF.join(predictionsPDF)
scored_pointsPDF.head()

- Plot the clusters and cluster centers.

In [ ]:
def plot_clusters(centers, clustered_pointsPDF):
    colors = ['orange', 'yellow', 'green', 'blue', 'purple']
    m = folium.Map(location=[37.62, -122.365], zoom_start=7.5)

    for index, point in clustered_pointsPDF.iterrows():
        folium.CircleMarker(location=[point['LAT'], point['LON']], 
                          color=colors[int(point['prediction'])],
                          radius=1) \
              .add_to(m)
        
    for (i, center) in enumerate(centers):
        folium.CircleMarker(location=center.tolist(),color='red') \
              .add_to(m)

    return(m)


In [ ]:
plot_clusters(kmeans.to_sklearn().cluster_centers_, scored_pointsPDF)

- How many customers are served by each hub?

In [ ]:
for i in range(k):
    n = scored_pointsPDF[scored_pointsPDF["prediction"]==i].shape[0]
    print (i, n)

### Save Cluster Centers to Snowflake

In [ ]:
# Create Pandas DataFrame to save
centersPDF = pd.DataFrame(kmeans.to_sklearn().cluster_centers_, columns=['LAT', 'LON'])
centersPDF.reset_index(inplace=True)
centersPDF = centersPDF.rename({'index': 'CLUSTER_ID'}, axis=1)

- Create Snowflake schema in which to save cluster centers

In [ ]:
session.sql('create or replace schema clustering').collect()

- Save as table **CLUSTER_CENTERS**.

In [ ]:
# Save to a Snowflake table
session.write_pandas(centersPDF, 'CLUSTER_CENTERS', schema='CLUSTERING', auto_create_table=True)

### Areas for Further Investigation/Work
* Find a model that groups the members into four clusters. Plot the results.
* Try five clusters.
* Can you think of additional questions to pursue?